# Step 2 — PI-CAI Fold 0 İndirme ve Veri Keşfi

**Amaç:** PI-CAI Public Training fold 0 (~5.4 GB) → indir, aç, görselleştir, 20-hastalık subset hazırla.

## 🛡️ Tasarım Prensibi: Idempotent
**Her hücre tekrar çalıştırılabilir** — durumu kontrol eder, zaten yapıldıysa atlar, eksikse tamamlar. Colab disconnect / kernel reset olsa bile sıfırdan başlamak gerekmez.

## Çalıştırma Sırası
1. Runtime → Change runtime type → **T4 GPU**
2. Hücreleri sırayla **Shift+Enter** ile çalıştır
3. Hata olursa: durdur, çıktıyı paylaş

## Beklenen Çıktılar
- `output/figures/sample_case_sequences.png` (sunum slaytı 4)
- `output/figures/sample_case_gt_overlays.png` (sunum slaytı 5)
- `output/metrics/subset20_inventory.csv` (20-hasta listesi)
- `input/images/subset20/` (Drive'a kalıcılaştırılmış subset)

## 0 — Drive bağla + Path'ler

In [ ]:
from google.colab import drive
if not __import__('os').path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive zaten bağlı.')

from pathlib import Path

# Drive yolları (kalıcı)
PROJECT_ROOT  = Path('/content/drive/MyDrive/Prostate_MRI_Project')
INPUT_DIR     = PROJECT_ROOT / 'input'
IMAGES_DIR    = INPUT_DIR / 'images'
LABELS_DIR    = INPUT_DIR / 'picai_labels'
OUTPUT_DIR    = PROJECT_ROOT / 'output'
FIGURES_DIR   = OUTPUT_DIR / 'figures'
METRICS_DIR   = OUTPUT_DIR / 'metrics'
DRIVE_SUBSET  = IMAGES_DIR / 'subset20'

# Colab local disk (geçici, hızlı)
LOCAL_WORK    = Path('/content/picai_work')
LOCAL_ZIP     = LOCAL_WORK / 'fold0.zip'
LOCAL_IMAGES  = LOCAL_WORK / 'images'

for d in [INPUT_DIR, IMAGES_DIR, LABELS_DIR, OUTPUT_DIR, FIGURES_DIR, METRICS_DIR,
          LOCAL_WORK, LOCAL_IMAGES, DRIVE_SUBSET]:
    d.mkdir(parents=True, exist_ok=True)

ZENODO_URL = 'https://zenodo.org/api/records/6624726/files/picai_public_images_fold0.zip/content'

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'LOCAL_WORK:   {LOCAL_WORK}')

## 1 — Paket kontrolü ve kurulum (eksik varsa)

Kernel reset / VM reassignment olduysa paketler uçmuş olabilir. Bu hücre **eksik olanları tespit edip kurar**, varsa atlar.

In [ ]:
import importlib, sys, subprocess

required = {
    'SimpleITK':   'SimpleITK',
    'nibabel':     'nibabel',
    'matplotlib':  'matplotlib',
    'pandas':      'pandas',
    'monai':       'monai[all]==1.3.2',
    'requests':    'requests',
}

missing = []
for mod, pkg in required.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f'Eksik paketler kuruluyor: {missing}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    print('\n✓ Kurulum tamam. Aşağıdaki uyarılar normal (Colab default paketleri için, biz kullanmıyoruz).')
else:
    print('✓ Tüm paketler kurulu.')

# Doğrula
import SimpleITK as sitk, nibabel as nib, matplotlib, pandas as pd, numpy as np, monai
print()
print(f'SimpleITK:  {sitk.Version_VersionString()}')
print(f'nibabel:    {nib.__version__}')
print(f'matplotlib: {matplotlib.__version__}')
print(f'pandas:     {pd.__version__}')
print(f'numpy:      {np.__version__}')
print(f'MONAI:      {monai.__version__}')

## 2 — Disk durumu

In [ ]:
!df -h /content /content/drive/MyDrive 2>/dev/null

## 3 — Zenodo'dan fold 0'ı indir (yoksa) 

**Idempotent:** Zip lokal'de tamsa atlar. Eksikse `curl -C -` ile resume eder.

In [ ]:
import requests
import os

def check_zip_status():
    """Zip durumunu: yok / eksik / tam olarak döndürür."""
    if not LOCAL_ZIP.exists():
        return 'absent', 0, 0
    local = LOCAL_ZIP.stat().st_size
    try:
        r = requests.head(ZENODO_URL, allow_redirects=True, timeout=30)
        expected = int(r.headers.get('content-length', 0))
    except Exception as e:
        print(f'Zenodo HEAD başarısız ({e}), boyut karşılaştırılamadı.')
        return 'unknown', local, 0
    if local == expected:
        return 'complete', local, expected
    elif local < expected:
        return 'partial', local, expected
    else:
        return 'oversized', local, expected

status, local, expected = check_zip_status()
print(f'Lokal:    {local/1e9:.3f} GB ({local:,} bytes)')
print(f'Beklenen: {expected/1e9:.3f} GB ({expected:,} bytes)')
print(f'Durum:    {status}\n')

if status == 'complete':
    print('✓ Zip tam — indirme atlanıyor.')
elif status in ('absent', 'partial'):
    print('İndirme başlıyor (5-20 dk)... `curl -C -` resume yapar.')
    os.chdir(LOCAL_WORK)
    !curl -C - -L "{ZENODO_URL}" -o "{LOCAL_ZIP}"
    # Tekrar kontrol
    status, local, expected = check_zip_status()
    print(f'\nİndirme sonrası: {status} ({local/1e9:.3f} / {expected/1e9:.3f} GB)')
    if status != 'complete':
        raise RuntimeError(f'İndirme tamamlanamadı (status={status}). Hücreyi tekrar çalıştır.')
else:
    print(f'⚠️ Beklenmedik durum: {status}. Manuel kontrol gerekli.')

## 4 — Unzip (yapılmamışsa)

**Idempotent:** Zaten açılmışsa atlar.

In [ ]:
# Açıldı mı? İçerikte .mha dosyası var mı diye bak
existing_mha = list(LOCAL_IMAGES.rglob('*.mha'))
existing_dirs = [d for d in LOCAL_IMAGES.iterdir() if d.is_dir()] if LOCAL_IMAGES.exists() else []

if len(existing_mha) > 100 or len(existing_dirs) > 50:
    print(f'✓ Zaten açılmış: {len(existing_dirs)} klasör, {len(existing_mha)} .mha dosyası → unzip atlanıyor.')
else:
    print('Unzip başlıyor (5-10 dk)...')
    !unzip -q -o "{LOCAL_ZIP}" -d "{LOCAL_IMAGES}"
    existing_mha = list(LOCAL_IMAGES.rglob('*.mha'))
    existing_dirs = [d for d in LOCAL_IMAGES.iterdir() if d.is_dir()]
    print(f'\n✓ Unzip tamam: {len(existing_dirs)} klasör, {len(existing_mha)} .mha dosyası')

## 5 — Hasta envanteri

Klasör yapısını anla, hasta ID listesini çıkar.

In [ ]:
import re

# PI-CAI iki olası yapı: (a) hasta_klasörü/dosya.mha  veya  (b) flat dosya.mha
patient_dirs = sorted([d for d in LOCAL_IMAGES.iterdir() if d.is_dir()])

if patient_dirs:
    structure = 'nested'
    print(f'Yapı: nested (hasta klasörleri)')
    print(f'Hasta klasörü sayısı: {len(patient_dirs)}')
    print(f'İlk 5: {[d.name for d in patient_dirs[:5]]}')
    sample_case_dir = patient_dirs[0]
    sample_files = sorted(sample_case_dir.glob('*.mha'))
    print(f'\nÖrnek hasta ({sample_case_dir.name}):')
    for f in sample_files:
        print(f'  {f.name:40s} {f.stat().st_size/1e6:6.1f} MB')
else:
    structure = 'flat'
    all_mha = sorted(LOCAL_IMAGES.rglob('*.mha'))
    print(f'Yapı: flat')
    print(f'.mha dosya sayısı: {len(all_mha)}')
    # patient_id_study_id_seq.mha → patient_id_study_id
    case_ids = sorted(set(re.match(r'(\d+_\d+)', p.name).group(1)
                          for p in all_mha if re.match(r'(\d+_\d+)', p.name)))
    print(f'Eşsiz hasta_id: {len(case_ids)}')
    print(f'İlk 5: {case_ids[:5]}')

print(f'\nstructure = {structure!r}')

## 6 — Örnek hasta: T2W + ADC + HBV görselleştir

**Slayt 4** için: 3 sekansı yan yana göster.

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt

def find_seq(files, suffix):
    """_t2w.mha gibi suffix'i ara."""
    for f in files:
        if f.name.lower().endswith(f'_{suffix}.mha'):
            return f
    return None

def load_mha(path):
    if path is None or not path.exists():
        return None, None
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)  # (Z, Y, X)
    return arr, img

# Örnek hasta seç
if structure == 'nested':
    sample_case_id = patient_dirs[0].name
    sample_files = list(patient_dirs[0].glob('*.mha'))
else:
    sample_case_id = case_ids[0]
    sample_files = list(LOCAL_IMAGES.rglob(f'{sample_case_id}*.mha'))

t2w_path = find_seq(sample_files, 't2w')
adc_path = find_seq(sample_files, 'adc')
hbv_path = find_seq(sample_files, 'hbv')

print(f'Hasta: {sample_case_id}')
print(f'  T2W: {t2w_path.name if t2w_path else "YOK"}')
print(f'  ADC: {adc_path.name if adc_path else "YOK"}')
print(f'  HBV: {hbv_path.name if hbv_path else "YOK"}')

t2w, t2w_img = load_mha(t2w_path)
adc, _       = load_mha(adc_path)
hbv, _       = load_mha(hbv_path)

if t2w is not None:
    print(f'\nT2W shape: {t2w.shape}, spacing: {t2w_img.GetSpacing()}')
    print(f'T2W range: [{t2w.min()}, {t2w.max()}]')

In [ ]:
# 3 sequence yan yana, orta dilim
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, vol) in zip(axes, [('T2W', t2w), ('ADC', adc), ('HBV (DWI)', hbv)]):
    if vol is None:
        ax.text(0.5, 0.5, f'{name}: yok', ha='center', va='center')
        ax.axis('off')
        continue
    mid = vol.shape[0] // 2
    ax.imshow(vol[mid], cmap='gray')
    ax.set_title(f'{name}  (slice {mid+1}/{vol.shape[0]})')
    ax.axis('off')

plt.suptitle(f'PI-CAI Multi-parametric MRI — Case {sample_case_id}', fontsize=14)
plt.tight_layout()

fig_path = FIGURES_DIR / 'sample_case_sequences.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'✓ Kaydedildi: {fig_path}')
plt.show()

## 7 — Ground Truth overlay (slayt 5)

Aynı hastanın anatomi (whole gland, zonal) + lezyon (human_expert) maskelerini T2W üzerine bindir.

In [ ]:
import nibabel as nib

def find_label(pattern_dir, case_id):
    """`case_id` ile başlayan .nii.gz dosyalarını ara."""
    if not pattern_dir.exists():
        return None
    matches = list(pattern_dir.rglob(f'{case_id}*.nii.gz'))
    return matches[0] if matches else None

def load_nii(path):
    if path is None or not path.exists():
        return None
    # nibabel'in (X,Y,Z) düzenini SITK'nin (Z,Y,X) düzenine çevir
    return nib.load(str(path)).get_fdata().transpose(2, 1, 0)

# 3 label tipini ara
lesion_path = find_label(LABELS_DIR / 'csPCa_lesion_delineations' / 'human_expert' / 'resampled', sample_case_id)
if lesion_path is None:
    lesion_path = find_label(LABELS_DIR / 'csPCa_lesion_delineations' / 'human_expert' / 'original', sample_case_id)
wg_path     = find_label(LABELS_DIR / 'anatomical_delineations' / 'whole_gland' / 'AI', sample_case_id)
zonal_path  = find_label(LABELS_DIR / 'anatomical_delineations' / 'zonal_pz_tz' / 'AI', sample_case_id)

print(f'Lezyon (human):  {lesion_path}')
print(f'Whole gland AI:  {wg_path}')
print(f'Zonal AI:        {zonal_path}')

lesion_mask = load_nii(lesion_path)
wg_mask     = load_nii(wg_path)
zonal_mask  = load_nii(zonal_path)

In [ ]:
# T2W üzerine 3 farklı mask overlay
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
mid = t2w.shape[0] // 2

panels = [
    ('Whole Gland (AI)',     wg_mask,     'autumn'),
    ('Zonal PZ (=2) / TZ (=1) (AI)', zonal_mask, 'viridis'),
    ('csPCa Lesion (Human)', lesion_mask, 'Reds'),
]

for ax, (name, mask, cmap) in zip(axes, panels):
    ax.imshow(t2w[mid], cmap='gray')
    if mask is not None and mid < mask.shape[0]:
        masked = np.ma.masked_where(mask[mid] == 0, mask[mid])
        ax.imshow(masked, cmap=cmap, alpha=0.5)
        ax.set_title(f'{name}\n(slice {mid+1}/{t2w.shape[0]})')
    else:
        ax.set_title(f'{name}\n(label yok bu hasta için)')
    ax.axis('off')

plt.suptitle(f'Ground Truth Overlays — Case {sample_case_id}', fontsize=14)
plt.tight_layout()
fig_path = FIGURES_DIR / 'sample_case_gt_overlays.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'✓ Kaydedildi: {fig_path}')
plt.show()

## 8 — Marksheet + 20-hasta subset seçimi

10 csPCa-positive + 10 csPCa-negative. Drive'a kalıcılaştır.

In [ ]:
import pandas as pd

marksheet_path = LABELS_DIR / 'clinical_information' / 'marksheet.csv'
marksheet = pd.read_csv(marksheet_path)
marksheet['patient_str'] = marksheet['patient_id'].astype(str)

print(f'Marksheet: {len(marksheet)} hasta')
print(f'Sütunlar: {list(marksheet.columns)}')
marksheet.head()

In [ ]:
# Lokal hasta_id'lerini al
if structure == 'nested':
    local_patient_ids = sorted(set(d.name.split('_')[0] for d in patient_dirs))
else:
    local_patient_ids = sorted(set(c.split('_')[0] for c in case_ids))

print(f'Lokal\'de toplam: {len(local_patient_ids)} eşsiz hasta')

# Marksheet ile kesişim
local_in_marksheet = marksheet[marksheet['patient_str'].isin(local_patient_ids)]
print(f'Marksheet ile eşleşen: {len(local_in_marksheet)}')

# csPCa pozitif/negatif: case_csPCa sütununu kullan (YES/NO)
pos = local_in_marksheet[local_in_marksheet['case_csPCa'] == 'YES'].head(10)
neg = local_in_marksheet[local_in_marksheet['case_csPCa'] == 'NO'].head(10)

print(f'\nPozitif (csPCa YES) seçildi: {len(pos)}')
print(f'Negatif (csPCa NO)  seçildi: {len(neg)}')
print(f'\nPozitif IDs: {list(pos["patient_str"])}')
print(f'Negatif IDs: {list(neg["patient_str"])}')

subset_ids = list(pos['patient_str']) + list(neg['patient_str'])

In [ ]:
# Subset'i Drive'a kopyala (idempotent — varsa atlar)
import shutil

copied_new = 0
for pid in subset_ids:
    if structure == 'nested':
        matches = [d for d in patient_dirs if d.name.startswith(pid)]
        for src in matches:
            dst = DRIVE_SUBSET / src.name
            if not dst.exists():
                shutil.copytree(src, dst)
                copied_new += 1
    else:
        srcs = list(LOCAL_IMAGES.rglob(f'{pid}*.mha'))
        case_dir = DRIVE_SUBSET / pid
        case_dir.mkdir(exist_ok=True)
        for src in srcs:
            dst = case_dir / src.name
            if not dst.exists():
                shutil.copy(src, dst)
                copied_new += 1

print(f'Yeni kopyalanan: {copied_new} dosya/klasör')
print(f'\nDrive subset durumu:')
!du -sh "{DRIVE_SUBSET}"
!ls "{DRIVE_SUBSET}" | head -25

## 9 — Subset envanter CSV (sunum için)

In [ ]:
subset_info = pd.concat([
    pos.assign(group='positive'),
    neg.assign(group='negative'),
], ignore_index=True)

show_cols = ['patient_id', 'study_id', 'group', 'case_csPCa', 'case_ISUP',
             'patient_age', 'psa', 'prostate_volume', 'center']
show_cols = [c for c in show_cols if c in subset_info.columns]
subset_summary = subset_info[show_cols]

out_csv = METRICS_DIR / 'subset20_inventory.csv'
subset_summary.to_csv(out_csv, index=False)
print(f'✓ Envanter CSV kaydedildi: {out_csv}\n')
subset_summary

---

## ✅ Step 2 Tamamlandı

**Üretilenler:**
- 📊 `output/figures/sample_case_sequences.png` — T2W/ADC/HBV (slayt 4)
- 🎨 `output/figures/sample_case_gt_overlays.png` — Ground truth overlays (slayt 5)
- 📋 `output/metrics/subset20_inventory.csv` — 10 pozitif + 10 negatif hasta listesi
- 💾 `input/images/subset20/` — Drive'a kalıcılaştırılmış imaging veri

**Sonraki:** `step3_anatomy_inference.ipynb` — MONAI bundle ile zonal segmentasyon inference.

---

### 🆘 Kernel reset / disconnect olursa
Bu notebook **idempotent** — sadece tepeden tırnağa yeniden çalıştır. Tüm hücreler durumu kontrol edip ya atlar ya tamamlar. İndirme, unzip, kopyalama hepsi cache'lenir.